# A form generated from a spec

An app on a client: `ess.apps.panel.App` generates a form for each spec from its pydantic models, with a tab per parameter and a widget for each field, at any depth, that follows from the field's type (`ess.spec.field_types.split`): an input for a scalar, a picker for a file field, a choice of ways for a group, a union, `Referenced[T]`, or `X | None`, a list of inputs for a list of scalars, and a table, edited a row at a time, for any other list.
Every `X | None` is a choice with a way `None`, a file field's too, and a choice or a table at a field of a group shows in a card titled by the field.
The app submits each request through the client, shows each reason the backend refuses it at its position, and why a record failed.
It picks files from `client.files`, and, at each field typed `Referenced[T]`, references from the client's records whose outputs fit `T` and that the client may read.
In a tab of its own, `ess.apps.panel.Outputs` lists the outputs of the records by type, in a sidebar, each as `BeamCentre (11:45)`, the local time its record was created: what it is, its value in brief, and its plot or its provenance graph.
A third tab lists the records themselves.

The specs are those of `toy_sans.py`.

In [ ]:
import tempfile
from pathlib import Path

import panel as pn

import toy_sans as t
from ess.apps import local
from ess.apps.panel import App
from ess.apps.testing import FakeFiles

pn.extension('tabulator')

## The app

The client is that of a backend in this process, `local()`, with the bindings of the specs.
Its file source holds the toy runs, each with its role.
Try: find a beam centre in run 101, and a transmission from run 120.
Then the form of I(Q) offers each as a record.
The form of I(Q) of summed runs takes the sample and can runs as tables, with a row per run: pick a run in the editor below the table, and click Add as new row.
A table starts with nothing given, until a row is added or removed.

A submission returns at once, and the record is listed pending.
The app asks the client what changed twice a second (`client.changes`), so it shows a new or finished record of any client of the proposal.
Counts per run takes a second per run, long enough to see a record pending: give it runs 101, 102, and 103, and select its output, which is plotted once it has completed.
Counts cube outputs a 4-D array of about 320 MB (50 x 20 x 200 x 200 values): select its output, and the plot has a slider for time and layer, and reads only the cut shown.
Pick Provenance to see the records and files the output came from, coloured by status; click a node to see a record's spec and params, or a file's metadata.
Remove releases the output's record, which drops the value of each of its outputs; the record stays in the graphs of the records that read it.

The cell shows the app here, and `panel serve spec-form.ipynb --show`, run in this folder, serves it as a page of its own.

In [ ]:
files = FakeFiles('p1', Path(tempfile.mkdtemp()))
for runs, role in (
    (t.SAMPLE_RUNS, 'sample'),
    (t.CAN_RUNS, 'background'),
    (t.TRANSMISSION_RUNS, 'transmission'),
):
    for run, content in runs.items():
        files.measure(run, content, role=role)

bindings = [
    t.find_centre_binding,
    t.transmission_binding,
    t.single,
    t.multi,
    t.forward_binding,
    t.counts_binding,
    t.cube_binding,
]
client = local(proposal='p1', files=files, bind=bindings)
app = App(client)
app.panel().servable()